In [5]:
!pip install pytorch-tabnet -q

Error processing line 1 of /Users/tonieenriquez/anaconda3/lib/python3.11/site-packages/distutils-precedence.pth:

  Traceback (most recent call last):
    File "<frozen site>", line 186, in addpackage
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named '_distutils_hack'

Remainder of file ignored

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [6]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find repo root containing /src")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("torch       =", torch.__version__)

from pytorch_tabnet.tab_model import TabNetClassifier

from src.evaluation import evaluate_binary_probabilities
from src.models.feature_sets import (
    BASE_TABULAR_FEATURES,
    GCMT_FEATURES,
    QUALITY_FEATURES,
)
from src.models.input_layer import (
    InputConfig,
    load_modeling_splits,
    prepare_tabular_inputs,
)
from src.utils.paths import METRICS_DIR

PROJECT_ROOT = /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10
torch       = 2.10.0


In [7]:
DATASET_NAME = "earthquake_aftershock_v2_gcmt"
MODEL_NAME = "tabnet_best_v1"
TARGETS = ["y_24h", "y_72h"]

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [8]:
splits = load_modeling_splits(dataset_name=DATASET_NAME)

for split_name, df in splits.items():
    print(
        f"{split_name:5s}  {len(df):>5} rows  |  "
        f"y_24h pos: {df['y_24h'].mean():.3f}  "
        f"y_72h pos: {df['y_72h'].mean():.3f}"
    )

train  23031 rows  |  y_24h pos: 0.443  y_72h pos: 0.504
val     1744 rows  |  y_24h pos: 0.478  y_72h pos: 0.535
test    3513 rows  |  y_24h pos: 0.469  y_72h pos: 0.530


In [9]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    has_gcmt = df["has_gcmt"].fillna(False) if "has_gcmt" in df.columns else pd.Series(False, index=df.index)

    # Depth regime
    if "trigger_depth_km" in df.columns:
        df["depth_shallow"] = (df["trigger_depth_km"] < 70).astype(float)
        df["depth_intermediate"] = (df["trigger_depth_km"].between(70, 300)).astype(float)
        df["depth_deep"] = (df["trigger_depth_km"] > 300).astype(float)

    # Magnitude / activity transforms
    if "trigger_magnitude" in df.columns:
        df["log_magnitude"] = np.log10(df["trigger_magnitude"].clip(lower=1e-6))

    if "trigger_magnitude" in df.columns and "trigger_depth_km" in df.columns:
        df["mag_depth_ratio"] = df["trigger_magnitude"] / (df["trigger_depth_km"] + 1)

    if "prior_global_event_count_24h" in df.columns:
        df["log_prior_24h"] = np.log1p(df["prior_global_event_count_24h"])

    if "prior_global_event_count_7d" in df.columns:
        df["log_prior_7d"] = np.log1p(df["prior_global_event_count_7d"])

    if "prior_global_event_count_24h" in df.columns and "prior_global_event_count_7d" in df.columns:
        denom = (df["prior_global_event_count_7d"] / 7.0).replace(0, np.nan)
        df["seismicity_acceleration"] = df["prior_global_event_count_24h"] / denom

    if "trigger_magnitude" in df.columns and "prior_global_event_count_24h" in df.columns:
        df["mag_x_log_prior"] = df["trigger_magnitude"] * np.log1p(df["prior_global_event_count_24h"])

    if "trigger_magnitude" in df.columns and "depth_shallow" in df.columns:
        df["mag_x_shallow"] = df["trigger_magnitude"] * df["depth_shallow"]

    # Time cyclicals
    if "trigger_month" in df.columns:
        df["sin_month"] = np.sin(2 * np.pi * df["trigger_month"] / 12)
        df["cos_month"] = np.cos(2 * np.pi * df["trigger_month"] / 12)

    if "trigger_hour" in df.columns:
        df["sin_hour"] = np.sin(2 * np.pi * df["trigger_hour"] / 24)
        df["cos_hour"] = np.cos(2 * np.pi * df["trigger_hour"] / 24)

    if "trigger_dayofyear" in df.columns:
        df["sin_dayofyear"] = np.sin(2 * np.pi * df["trigger_dayofyear"] / 365)
        df["cos_dayofyear"] = np.cos(2 * np.pi * df["trigger_dayofyear"] / 365)

    # Spatial
    if "trigger_longitude" in df.columns and "trigger_latitude" in df.columns:
        df["ring_of_fire"] = (
            (np.abs(df["trigger_longitude"]) > 130)
            & (df["trigger_latitude"].between(-60, 60))
        ).astype(float)

    # GCMT-enriched features
    if "gcmt_scalar_moment" in df.columns:
        df["log_scalar_moment"] = np.where(
            has_gcmt & df["gcmt_scalar_moment"].notna(),
            np.log10(df["gcmt_scalar_moment"].clip(lower=1e-10)),
            np.nan,
        )

    if "gcmt_moment_exponent" in df.columns:
        df["moment_exponent_centered"] = (df["gcmt_moment_exponent"] - 24.0).where(has_gcmt)

    if "gcmt_scalar_moment" in df.columns and "trigger_magnitude" in df.columns:
        df["gcmt_mw"] = np.where(
            has_gcmt & df["gcmt_scalar_moment"].notna(),
            (2 / 3) * np.log10(df["gcmt_scalar_moment"].clip(lower=1e-10)) - 10.7,
            np.nan,
        )
        df["mw_trigger_diff"] = (df["gcmt_mw"] - df["trigger_magnitude"]).where(has_gcmt)

    if "dip" in df.columns:
        df["sin_dip"] = np.sin(np.radians(df["dip"].where(has_gcmt)))

    if {"gcmt_eig1", "gcmt_eig2", "gcmt_eig3"}.issubset(df.columns):
        e1 = df["gcmt_eig1"].where(has_gcmt)
        e2 = df["gcmt_eig2"].where(has_gcmt)
        e3 = df["gcmt_eig3"].where(has_gcmt)
        denom = (e1.abs() + e3.abs()).replace(0, np.nan)
        df["clvd_fraction"] = (2 * e2.abs() / denom).where(has_gcmt)
        df["eig_ratio"] = (e1.abs() / denom).where(has_gcmt)

    if {"gcmt_eig1_plunge", "gcmt_eig3_plunge"}.issubset(df.columns):
        p1 = df["gcmt_eig1_plunge"].where(has_gcmt)
        p3 = df["gcmt_eig3_plunge"].where(has_gcmt)
        df["sin_eig1_plunge"] = np.sin(np.radians(p1))
        df["cos_eig1_plunge"] = np.cos(np.radians(p1))
        df["sin_eig3_plunge"] = np.sin(np.radians(p3))
        df["cos_eig3_plunge"] = np.cos(np.radians(p3))
        df["tp_plunge_diff"] = (p1 - p3).where(has_gcmt)

    if "gcmt_depth_km" in df.columns and "trigger_depth_km" in df.columns:
        df["centroid_depth_diff"] = (df["gcmt_depth_km"] - df["trigger_depth_km"]).where(has_gcmt)

    if "gcmt_half_duration_sec" in df.columns:
        df["log_half_duration"] = np.log1p(df["gcmt_half_duration_sec"].where(has_gcmt))

    if "gcmt_mag_diff" in df.columns:
        df["mag_diff_abs"] = df["gcmt_mag_diff"].abs().where(has_gcmt)

    if "rake" in df.columns:
        def _rake_regimes(rake: float):
            if pd.isna(rake):
                return np.nan, np.nan, np.nan
            r = rake % 360
            ss_dist = min(abs(r), abs(r - 180), abs(r - 360))
            return float(ss_dist < 45), float(45 <= r <= 135), float(225 <= r <= 315)

        regimes = df["rake"].apply(_rake_regimes)
        df["is_strike_slip"] = regimes.apply(lambda x: x[0]).where(has_gcmt)
        df["is_reverse"] = regimes.apply(lambda x: x[1]).where(has_gcmt)
        df["is_normal"] = regimes.apply(lambda x: x[2]).where(has_gcmt)

    return df

splits_eng = {name: engineer_features(df) for name, df in splits.items()}
print("Feature engineering complete.")
print("Columns after engineering:", splits_eng["train"].shape[1])

Feature engineering complete.
Columns after engineering: 127


In [10]:
ENGINEERED_FEATURES = [
    "depth_shallow", "depth_intermediate", "depth_deep",
    "log_magnitude", "mag_depth_ratio",
    "log_prior_24h", "log_prior_7d", "seismicity_acceleration",
    "mag_x_log_prior", "mag_x_shallow",
    "sin_month", "cos_month",
    "sin_hour", "cos_hour",
    "sin_dayofyear", "cos_dayofyear",
    "ring_of_fire",
    "log_scalar_moment", "moment_exponent_centered",
    "gcmt_mw", "mw_trigger_diff",
    "sin_dip",
    "clvd_fraction", "eig_ratio",
    "sin_eig1_plunge", "cos_eig1_plunge",
    "sin_eig3_plunge", "cos_eig3_plunge",
    "tp_plunge_diff",
    "centroid_depth_diff",
    "log_half_duration",
    "mag_diff_abs",
    "is_strike_slip", "is_reverse", "is_normal",
]

_seen = set()
FULL_FEATURE_SET = []

for f in (BASE_TABULAR_FEATURES + QUALITY_FEATURES + GCMT_FEATURES + ENGINEERED_FEATURES):
    if f not in _seen:
        FULL_FEATURE_SET.append(f)
        _seen.add(f)

print("Total features:", len(FULL_FEATURE_SET))

missing_engineered = [c for c in ENGINEERED_FEATURES if c not in splits_eng["train"].columns]
print("Missing engineered features:", missing_engineered)

Total features: 91
Missing engineered features: []


In [11]:
prepared = {}

for target in TARGETS:
    config = InputConfig(
        feature_cols=FULL_FEATURE_SET,
        target_col=target,
        missing_strategy="none",
        scale=False,
        allow_missing_optional=True,
        drop_rows_with_missing_target=True,
    )

    inputs = prepare_tabular_inputs(config=config, splits=splits_eng)
    prepared[target] = inputs
    print(f"{target} — X_train: {inputs.X_train.shape}")

y_24h — X_train: (23031, 91)
y_72h — X_train: (23031, 91)


In [12]:
def make_tabnet_arrays(inp):
    imputer = SimpleImputer(strategy="median")

    X_train = imputer.fit_transform(inp.X_train).astype(np.float32)
    X_val   = imputer.transform(inp.X_val).astype(np.float32)
    X_test  = imputer.transform(inp.X_test).astype(np.float32)

    y_train = np.asarray(inp.y_train).astype(int)
    y_val   = np.asarray(inp.y_val).astype(int)
    y_test  = np.asarray(inp.y_test).astype(int)

    return X_train, X_val, X_test, y_train, y_val, y_test, imputer

In [13]:
TABNET_PARAMS = dict(
    n_d=32,
    n_a=32,
    n_steps=4,
    gamma=1.5,
    n_independent=2,
    n_shared=2,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
    mask_type="entmax",
    scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
    scheduler_params=dict(
        mode="min",
        factor=0.5,
        patience=10,
        min_lr=1e-5,
    ),
    seed=SEED,
    verbose=1,
)

In [14]:
models = {}
artifacts = {}

for target in TARGETS:
    inp = prepared[target]
    X_train, X_val, X_test, y_train, y_val, y_test, imputer = make_tabnet_arrays(inp)

    clf = TabNetClassifier(**TABNET_PARAMS)

    clf.fit(
        X_train=X_train,
        y_train=y_train,
        eval_set=[(X_val, y_val)],
        eval_name=["val"],
        eval_metric=["logloss"],
        max_epochs=200,
        patience=25,
        batch_size=512,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False,
    )

    models[target] = clf
    artifacts[target] = {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
        "imputer": imputer,
        "raw": inp,
    }

    val_prob = clf.predict_proba(X_val)[:, 1]
    test_prob = clf.predict_proba(X_test)[:, 1]

    print(
        f"{target} — val AUC: {roc_auc_score(y_val, val_prob):.4f} | "
        f"test AUC: {roc_auc_score(y_test, test_prob):.4f}"
    )

epoch 0  | loss: 0.71018 | val_logloss: 0.93033 |  0:00:02s


TypeError: ReduceLROnPlateau.step() missing 1 required positional argument: 'metrics'